# MIDI Expression Pilot: Velocity and Articulation

This notebook tests one focused question: can controlled velocity and note-length variation make an explicitly composed MIDI phrase feel less mechanical?

## What stays fixed
- The pitch sequence, C-major scale, bar positions, tempo (96 BPM), meter (4/4), and note-onset grid.
- The bass part and its instrument patch.
- The renderer and SoundFont.

## What changes
- Baseline: the original fixed MIDI velocities and note lengths.
- Expressive: small metrical velocity accents, a gentle phrase-level dynamic curve, and slightly varied note lengths.

We render both versions with Acoustic Piano, Marimba, and Harp leads. Compare each pair using the same instrument. This is a renderer and MIDI-expression experiment; it does not train a generative model.

## Step 1 — Install or reuse FluidSynth

The SoundFont is installed only if this runtime does not already have it. The notebook needs no GPU, model weights, API key, or training data.

In [ ]:
from pathlib import Path
import shutil
import subprocess

SOUNDFONT = Path("/usr/share/sounds/sf2/FluidR3_GM.sf2")

def run(command, label):
    result = subprocess.run(command, capture_output=True, text=True)
    if result.returncode:
        print(result.stdout[-2000:])
        print(result.stderr[-4000:])
        raise RuntimeError(f"{label} failed with exit code {result.returncode}")
    return result

if not shutil.which("fluidsynth") or not SOUNDFONT.exists():
    print("Installing FluidSynth and FluidR3 GM...")
    run(["apt-get", "update", "-qq"], "apt-get update")
    run(["apt-get", "install", "-y", "-qq", "fluidsynth", "fluid-soundfont-gm"], "apt-get install")

if not shutil.which("fluidsynth") or not SOUNDFONT.exists():
    raise FileNotFoundError("FluidSynth or FluidR3_GM.sf2 is still missing.")

version = run(["fluidsynth", "--version"], "FluidSynth version").stdout.splitlines()[0]
print(version)
print("SoundFont:", SOUNDFONT)
print(f"SoundFont size: {SOUNDFONT.stat().st_size / (1024**2):.1f} MiB")

## Step 2 — Set the fixed musical controls

The phrase uses the same four-bar C-major motif as the timbre pilot. The three lead patches let us check whether expressive MIDI behavior works consistently across different sampled instruments.

In [ ]:
import hashlib
import json
import math
import struct
import zipfile
import numpy as np
from scipy.io.wavfile import read as read_wav
from IPython.display import Audio, display

OUT_DIR = Path("/content/midi_expression_pilot")
OUT_DIR.mkdir(parents=True, exist_ok=True)

BPM = 96
TONIC_MIDI = 60  # C4
SCALE_INTERVALS = [0, 2, 4, 5, 7, 9, 11]  # C major
METER = (4, 4)
BARS = 4
TICKS_PER_BEAT = 480
SAMPLE_RATE = 44100
LEAD_PRESETS = {"acoustic_piano": 0, "marimba": 12, "harp": 46}
BASS_PROGRAM = 32  # Acoustic Bass

MELODY = [
    [(0.0,0,92),(0.5,2,76),(1.0,4,84),(1.5,2,72),(2.0,3,86),(2.5,4,78),(3.0,2,74),(3.5,1,70)],
    [(0.0,4,88),(0.5,5,80),(1.0,6,84),(1.5,5,74),(2.0,4,86),(2.5,2,76),(3.0,3,78),(3.5,2,70)],
    [(0.0,0,90),(0.5,2,76),(1.0,4,84),(1.5,6,78),(2.0,5,86),(2.5,3,74),(3.0,4,80),(3.5,2,72)],
    [(0.0,1,84),(0.5,3,76),(1.0,5,82),(1.5,4,74),(2.0,2,84),(2.5,1,72),(3.0,0,80),(3.5,0,92)],
]
BASS_ROOT_DEGREES = [0, 3, 4, 2]

def degree_to_midi(degree, octave=0):
    size = len(SCALE_INTERVALS)
    octaves, index = divmod(degree, size)
    return TONIC_MIDI + 12 * (octaves + octave) + SCALE_INTERVALS[index]

lead_base, bass_events = [], []
for bar, notes in enumerate(MELODY):
    for beat_in_bar, degree, velocity in notes:
        lead_base.append({
            "bar": bar, "start_beat": bar * 4 + beat_in_bar,
            "beat_in_bar": beat_in_bar, "pitch": degree_to_midi(degree),
            "duration_beats": 0.80 if (bar == 3 and beat_in_bar == 3.5) else 0.42,
            "velocity": velocity,
        })
    root = BASS_ROOT_DEGREES[bar]
    for beat_in_bar, degree, velocity in [(0.0, root, 88), (2.0, root + 4, 76)]:
        bass_events.append({
            "bar": bar, "start_beat": bar * 4 + beat_in_bar,
            "pitch": degree_to_midi(degree, octave=-1),
            "duration_beats": 1.75, "velocity": velocity,
        })

print(f"{BARS} bars | {BPM} BPM | 4/4 | {len(lead_base)} lead notes | {len(bass_events)} bass notes")
print("Lead programs:", LEAD_PRESETS, "| Bass program:", BASS_PROGRAM)

## Step 3 — Create the two expression versions

The expressive version keeps every pitch and note-onset beat identical to the baseline. It changes only lead velocity and note length. Bass events are copied unchanged.

The changes are deliberately modest: stronger notes on important beats, a slight phrase contour, and shorter off-beat notes. This tests whether MIDI expression helps without loosening note, scale, or rhythm control.

In [ ]:
def make_expressive(events):
    result = []
    phrase_curve = [0, 2, 4, 1]  # subtle rise, then release
    for event in events:
        e = dict(event)
        beat = e["beat_in_bar"]
        metrical = 6 if beat == 0 else (3 if beat == 2 else (-2 if beat % 1 else 0))
        e["velocity"] = int(np.clip(e["velocity"] + metrical + phrase_curve[e["bar"]], 1, 120))
        if e["duration_beats"] < 0.7:
            e["duration_beats"] = 0.46 if beat in (0, 2) else 0.37
        result.append(e)
    return result

lead_expressive = make_expressive(lead_base)

assert [(e["pitch"], e["start_beat"]) for e in lead_base] == [
    (e["pitch"], e["start_beat"]) for e in lead_expressive
]
assert bass_events == [dict(e) for e in bass_events]
print("Pitch sequence and note-onset beats: identical")
print("Bass events: unchanged")
print("Lead velocities or note lengths changed:", sum(
    (a["velocity"], a["duration_beats"]) != (b["velocity"], b["duration_beats"])
    for a, b in zip(lead_base, lead_expressive)
), "of", len(lead_base))

## Step 4 — Write editable MIDI files

Each MIDI contains separate Lead and Bass tracks. The expressive and baseline files preserve the same tempo, meter, pitches, and note-onset beats. Keep the MIDI files: they are the editable musical source; WAV files are rendered previews.

In [ ]:
def vlq(value):
    value = int(value)
    if value < 0:
        raise ValueError("MIDI delta time cannot be negative")
    out = [value & 0x7F]
    value >>= 7
    while value:
        out.insert(0, (value & 0x7F) | 0x80)
        value >>= 7
    return bytes(out)

def meta(kind, payload):
    return bytes([0xFF, kind]) + vlq(len(payload)) + payload

def make_track(name, channel, note_events=None, program=None, tempo=None, meter=None):
    timed = [(0, -3, meta(0x03, name.encode("ascii")))]
    if tempo is not None:
        micros = round(60_000_000 / tempo)
        timed.append((0, -2, meta(0x51, micros.to_bytes(3, "big"))))
    if meter is not None:
        numerator, denominator = meter
        timed.append((0, -1, meta(0x58, bytes([numerator, int(math.log2(denominator)), 24, 8]))))
    if program is not None:
        timed.append((0, 0, bytes([0xC0 | channel, int(program)])))
    for e in note_events or []:
        start = round(e["start_beat"] * TICKS_PER_BEAT)
        end = round((e["start_beat"] + e["duration_beats"]) * TICKS_PER_BEAT)
        pitch = int(e["pitch"])
        velocity = int(np.clip(e["velocity"], 1, 127))
        timed.append((start, 1, bytes([0x90 | channel, pitch, velocity])))
        timed.append((end, 0, bytes([0x80 | channel, pitch, 0])))
    timed.sort(key=lambda item: (item[0], item[1]))
    body, previous = bytearray(), 0
    for tick, _, payload in timed:
        body.extend(vlq(tick - previous))
        body.extend(payload)
        previous = tick
    body.extend(b"\x00\xFF\x2F\x00")
    return b"MTrk" + len(body).to_bytes(4, "big") + body

def write_midi(path, lead_events, lead_program):
    tracks = [
        make_track("Conductor", 0, tempo=BPM, meter=METER),
        make_track("Lead", 0, lead_events, program=lead_program),
        make_track("Bass", 1, bass_events, program=BASS_PROGRAM),
    ]
    header = b"MThd" + (6).to_bytes(4, "big") + struct.pack(">HHH", 1, len(tracks), TICKS_PER_BEAT)
    Path(path).write_bytes(header + b"".join(tracks))
    return Path(path)

midi_files = {}
for preset_name, program in LEAD_PRESETS.items():
    for style, lead_events in [("baseline", lead_base), ("expressive", lead_expressive)]:
        name = f"{preset_name}_{style}"
        midi_files[name] = write_midi(OUT_DIR / f"{name}.mid", lead_events, program)
print("Created", len(midi_files), "editable MIDI arrangements.")

## Step 5 — Render the six controlled comparisons

The only differences within each instrument pair are the lead's velocity and note lengths. Bass, SoundFont, gain, and renderer settings remain fixed.

In [ ]:
wav_files = {}
for name, midi_path in midi_files.items():
    wav_path = OUT_DIR / f"{name}.wav"
    result = subprocess.run(
        ["fluidsynth", "-ni", "-g", "0.7", "-F", str(wav_path),
         "-r", str(SAMPLE_RATE), str(SOUNDFONT), str(midi_path)],
        capture_output=True, text=True,
    )
    if result.returncode or not wav_path.exists():
        print(result.stdout[-2000:])
        print(result.stderr[-4000:])
        raise RuntimeError(f"FluidSynth failed to render {name}")
    wav_files[name] = wav_path
    print("Rendered:", wav_path.name, f"({wav_path.stat().st_size / 1024:.0f} KiB)")

## Step 6 — Listen pair by pair

For each instrument, compare Baseline with Expressive. Focus on whether the melody still feels like the same phrase, whether accents sound intentional, and whether the longer/shorter notes improve the performance or make it uneven. The mixes are not loudness-normalized, so also notice if one version is simply louder overall.

In [ ]:
def listen_wav(path):
    sample_rate, audio = read_wav(str(path))
    if audio.dtype == np.int16:
        audio = audio.astype(np.float32) / 32768.0
    elif audio.dtype == np.int32:
        audio = audio.astype(np.float32) / 2147483648.0
    else:
        audio = audio.astype(np.float32)
        peak = float(np.max(np.abs(audio))) if audio.size else 0.0
        if peak > 1:
            audio = audio / peak
    display(Audio(audio.T if audio.ndim == 2 else audio, rate=sample_rate))

for preset in LEAD_PRESETS:
    print(f"{preset} — Baseline")
    listen_wav(wav_files[f"{preset}_baseline"])
    print(f"{preset} — Expressive")
    listen_wav(wav_files[f"{preset}_expressive"])

## Step 7 — Save the audio, MIDI, and experiment manifest

The ZIP contains the six rendered WAVs, six editable MIDI files, and a manifest describing the fixed and changed controls. It also includes the package copyright notices when available.

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

license_files = []
for source in [
    Path("/usr/share/doc/fluid-soundfont-gm/copyright"),
    Path("/usr/share/doc/fluidsynth/copyright"),
]:
    if source.exists():
        target = OUT_DIR / f"{source.parent.name}_copyright.txt"
        target.write_text(source.read_text(errors="replace"), encoding="utf-8")
        license_files.append(target.name)

manifest = {
    "experiment": "midi_expression_pilot_v1",
    "arrangement": {"key": "C major", "tempo_bpm": BPM, "meter": list(METER),
                    "bars": BARS, "sample_rate_hz": SAMPLE_RATE},
    "fixed_between_pairs": ["pitch sequence", "note-onset beats", "bass notes",
                            "bass patch", "SoundFont", "renderer gain"],
    "changed_in_expressive": ["lead velocities", "lead note lengths"],
    "lead_programs_zero_based_GM": LEAD_PRESETS,
    "bass_program_zero_based_GM": BASS_PROGRAM,
    "soundfont_sha256": sha256_file(SOUNDFONT),
    "soundfont_published_license": "MIT; package copyright notice included when available",
    "fluidsynth_license": "LGPL-2.1",
    "outputs": {"midi": {k: v.name for k, v in midi_files.items()},
                "wav": {k: v.name for k, v in wav_files.items()},
                "license_files": license_files},
}
manifest_path = OUT_DIR / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

bundle = Path("/content/midi_expression_pilot.zip")
with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for path in sorted(OUT_DIR.iterdir()):
        if path.is_file():
            z.write(path, arcname=path.name)
print("Bundle:", bundle, "| size:", f"{bundle.stat().st_size / (1024**2):.2f} MiB")
from google.colab import files
files.download(str(bundle))

## How to interpret the result

- If Expressive feels more natural while the melody and onset grid remain the same, MIDI velocity and articulation are useful product controls.
- If the change is barely audible, this SoundFont may not respond strongly enough to these controls; that does not mean note-level control is impossible.
- If it sounds worse, keep the explicit MIDI controls and allow users to adjust expression manually or use a better instrument library.
- This pilot does not yet test automatic composition, user interface, commercial redistribution of the SoundFont, or a trained neural music model.